# Public portfolio notebook
Outputs have been removed before publication. Execution requires authorised local course data. Saved aggregate results are in `reports/portfolio/`; the public dashboard uses synthetic data.


# BNPL cleaned-data EDA

This report uses the completed cleaning v1.2 outputs. Exploration covers train and internal
validation from 2021-02-28 through 2022-02-27. Protected test outcome distributions are excluded.
Only aggregate statistics and charts are printed; no individual customer or merchant rows appear.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/analysis_v1.json').exists())
sys.path.insert(0, str(root))
from src.bnpl_analysis.common import load_config
from src.run_artifacts import snapshot_manifest_path
config = load_config(root)
pd.set_option('display.max_columns', 12)


In [ ]:
from src.bnpl_analysis.eda import run_eda
eda_dir = root / 'reports/eda' / config['analysis_run_id']
summary_path = eda_dir / 'summary.json'
if not summary_path.exists():
    run_eda(root)
summary = json.loads(summary_path.read_text(encoding='utf-8'))
metrics = {k: v for k, v in summary.items() if not isinstance(v, dict)}
display(pd.DataFrame(metrics.items(), columns=['Metric', 'Value']))


## Amount distribution and monthly activity
Partial February periods should not be compared as complete months.

In [ ]:
display(pd.DataFrame(summary['amount_quantiles_aud'].items(), columns=['Quantile', 'Amount AUD']))
display(pd.read_json(eda_dir / 'monthly_aggregates.json'))


## Historical charts
Fraud coverage is the proportion matched to a signal, not a realised fraud rate.

In [ ]:
for path in sorted((eda_dir / 'figures').glob('*.png')):
    display(Image(filename=str(path)))


## Modelling implications

- Compare top-K value and simple scale baselines before a complex model.
- Preserve absent fraud observations and their coverage flags.
- Keep customer geography contextual; it is not merchant location.
- Recompute time-window features; the contributor's all-period merchant aggregate is not a predictor table.
- Fit all feature selection and preprocessing only on the training partition.
- See the configured run directories under `reports/eda/` and `reports/time_split/`.
